# E-Commerce Customer Churn Prediction & Analytics
### End-to-End Machine Learning Walkthrough (Assignment 2)

**Project Domain:** E-Commerce Telemetry & Behavioral Analytics  
**Objective:** Build, optimize, and deploy a machine learning system to predict customer churn (`0` = Retained, `1` = Churned).  
**Technological Stack:** Python, Pandas, NumPy, Scikit-learn, Imbalanced-learn, Matplotlib, Seaborn, Joblib  

---

## 1. Environment Setup & Core Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, classification_report
)
from imblearn.over_sampling import SMOTE

# Visual formatting settings
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
print("All libraries successfully imported!")

## 2. Dataset Loading & Exploration

In [ ]:
data_path = "../data/raw/ecommerce_churn_raw.csv"
df = pd.read_csv(data_path)

print(f"Total Records: {df.shape[0]} | Total Features: {df.shape[1]}")
df.head()

In [ ]:
# Inspect data types, null values, and memory usage
df.info()

## 3. Investigating Data Problems: Class Imbalance & Missing Values

In [ ]:
# Class distribution analysis
churn_counts = df['churn'].value_counts()
churn_pcts = df['churn'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Retained (0)', 'Churned (1)'], churn_counts.values, color=['#2b5c8f', '#d9534f'], width=0.5)
for bar, pct in zip(bars, churn_pcts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f"{pct:.1f}%", ha='center', fontweight='bold')
ax.set_title("Severe Class Imbalance in Raw Dataset")
plt.show()

# Missing value statistics
missing = df.isnull().sum()
print("Missing Values by Column:")
print(missing[missing > 0])

## 4. Leak-Free Preprocessing & Stratified Splitting
**Strict Data Leakage Rule:** We split raw data first. Imputation medians, IQR caps, and transformers are fitted **strictly on the training set**.

In [ ]:
X = df.drop(columns=['churn'])
y = df['churn']

# 80/20 Stratified Train-Test Split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train size: {len(X_train_raw)} records | Test size: {len(X_test_raw)} records")

In [ ]:
# Preprocessing functions
def clean_anomalies(data):
    d = data.copy()
    if 'customer_age' in d.columns:
        d.loc[d['customer_age'] < 18, 'customer_age'] = np.nan
    if 'total_spend' in d.columns:
        d.loc[d['total_spend'] < 0, 'total_spend'] = np.nan
    if 'payment_delay_days' in d.columns:
        d.loc[d['payment_delay_days'] < 0, 'payment_delay_days'] = 0
    if 'support_tickets' in d.columns:
        d.loc[d['support_tickets'] < 0, 'support_tickets'] = 0
    return d

# Fit imputer parameters strictly on train
cleaned_train = clean_anomalies(X_train_raw)
imputer_dict = {
    col: float(cleaned_train[col].median()) for col in cleaned_train.select_dtypes(include=[np.number]).columns
}
for cat in ['subscription_type', 'contract_type']:
    imputer_dict[cat] = cleaned_train[cat].mode().iloc[0]

def transform_missing(data, imputer):
    d = data.copy()
    for col, val in imputer.items():
        if col in d.columns:
            d[col] = d[col].fillna(val)
    return d

imputed_train = transform_missing(cleaned_train, imputer_dict)
imputed_test = transform_missing(clean_anomalies(X_test_raw), imputer_dict)

## 5. Domain Feature Engineering
We engineer 5 domain-informed behavioral indicators:
1. `average_historical_monthly_spend` = `total_spend / (tenure_months + 1)`
2. `spend_deviation` = `monthly_spend - average_historical_monthly_spend`
3. `engagement_score` = `login_frequency / (last_login_days + 1)`
4. `support_ticket_intensity` = `support_tickets / (tenure_months + 1)`
5. `payment_risk_score` = `payment_delay_days * support_tickets`

In [ ]:
def add_engineered_features(data):
    d = data.copy()
    d['average_historical_monthly_spend'] = (d['total_spend'] / (d['tenure_months'] + 1.0)).round(4)
    d['spend_deviation'] = (d['monthly_spend'] - d['average_historical_monthly_spend']).round(4)
    d['engagement_score'] = (d['login_frequency'] / (d['last_login_days'] + 1.0)).round(4)
    d['support_ticket_intensity'] = (d['support_tickets'] / (d['tenure_months'] + 1.0)).round(4)
    d['payment_risk_score'] = (d['payment_delay_days'] * d['support_tickets']).round(4)
    return d

feat_train = add_engineered_features(imputed_train)
feat_test = add_engineered_features(imputed_test)
print("Engineered Features successfully added!")
feat_train[['average_historical_monthly_spend', 'spend_deviation', 'engagement_score', 'payment_risk_score']].head()

## 6. Preprocessing ColumnTransformer & SMOTE Resampling

In [ ]:
num_cols = [
    'customer_age', 'tenure_months', 'monthly_spend', 'total_spend',
    'login_frequency', 'support_tickets', 'payment_delay_days', 'discount_used',
    'last_login_days', 'average_historical_monthly_spend', 'spend_deviation',
    'engagement_score', 'support_ticket_intensity', 'payment_risk_score'
]
cat_cols = ['subscription_type', 'contract_type']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

# Fit on train, transform train and test
X_train_trans = preprocessor.fit_transform(feat_train)
X_test_trans = preprocessor.transform(feat_test)

# Apply SMOTE strictly to Training Set
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_trans, y_train)
print(f"Post-SMOTE Balanced Training Records: {len(X_train_res)} ({y_train_res.mean():.1%} positive class)")

## 7. Model Training & Evaluation
We train:
1. **Baseline**: Logistic Regression
2. **Advanced**: Random Forest Classifier

In [ ]:
# Model 1: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_res, y_train_res)
lr_preds = lr_model.predict(X_test_trans)
lr_probs = lr_model.predict_proba(X_test_trans)[:, 1]

# Model 2: Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train_res, y_train_res)
rf_preds = rf_model.predict(X_test_trans)
rf_probs = rf_model.predict_proba(X_test_trans)[:, 1]

# Metrics summary
def print_metrics(name, y_true, y_pred, y_prob):
    print(f"=== {name} ===")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1-Score:  {f1_score(y_true, y_pred):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_true, y_prob):.4f}")
    print(f"PR-AUC:    {average_precision_score(y_true, y_prob):.4f}\n")

print_metrics("Baseline: Logistic Regression", y_test, lr_preds, lr_probs)
print_metrics("Tuned Random Forest", y_test, rf_preds, rf_probs)

## 8. Test Model Inference on Sample Payload

In [ ]:
# Load full saved production pipeline
pipeline = joblib.load("../models/best_churn_model.joblib")

sample_customer = pd.DataFrame([{
    'customer_age': 28,
    'tenure_months': 2,
    'monthly_spend': 140.0,
    'total_spend': 280.0,
    'login_frequency': 3,
    'support_tickets': 6,
    'payment_delay_days': 18,
    'subscription_type': 'Basic',
    'contract_type': 'Month-to-Month',
    'discount_used': 0,
    'last_login_days': 35
}])

pred = pipeline.predict(sample_customer)[0]
prob = pipeline.predict_proba(sample_customer)[0][1]

print(f"Prediction: {'Likely to Churn' if pred == 1 else 'Retained'}")
print(f"Churn Probability: {prob:.2%}")